In [1]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, recall_score

def clean_code(code):
    code = re.sub(r"/\*.*?\*/", " ", code, flags=re.DOTALL)
    code = re.sub(r"//[^\n]*", " ", code)
    code = re.sub(r"\s+", " ", code)
    return code.strip()

df = pd.read_json("D:/vuln-detection-ml/data/function.json")
df["clean"] = df["func"].apply(clean_code)

qemu = df[df["project"] == "qemu"]
ffmpeg = df[df["project"] == "FFmpeg"]
print(len(qemu), len(ffmpeg))

17549 9769


In [2]:
def evaluate(train_df, test_df, name):
    vec = TfidfVectorizer(
        token_pattern=r"[A-Za-z_][A-Za-z0-9_]*|\d+|==|!=|<=|>=|->|&&|\|\||[^\s\w]",
        lowercase=False, ngram_range=(1, 2), min_df=2, max_features=50000,
    )
    Xtr = vec.fit_transform(train_df["clean"])
    Xte = vec.transform(test_df["clean"])
    ytr, yte = train_df["target"], test_df["target"]

    models = {
        "Dummy": DummyClassifier(strategy="most_frequent"),
        "NB": MultinomialNB(alpha=0.01),
        "SVM": LinearSVC(C=0.5, class_weight="balanced", max_iter=5000),
    }
    for m_name, model in models.items():
        p = model.fit(Xtr, ytr).predict(Xte)
        print(f"{name} | {m_name}: acc={accuracy_score(yte, p):.3f}, "
              f"F1={f1_score(yte, p, zero_division=0):.3f}, "
              f"recall={recall_score(yte, p, zero_division=0):.3f}")
    print()

In [3]:
# cross-project
evaluate(qemu, ffmpeg, "QEMU -> FFmpeg")
evaluate(ffmpeg, qemu, "FFmpeg -> QEMU")

# within-project (80/20 split) for comparison
f_tr, f_te = train_test_split(ffmpeg, test_size=0.2, random_state=42, stratify=ffmpeg["target"])
evaluate(f_tr, f_te, "FFmpeg -> FFmpeg")

q_tr, q_te = train_test_split(qemu, test_size=0.2, random_state=42, stratify=qemu["target"])
evaluate(q_tr, q_te, "QEMU -> QEMU")

QEMU -> FFmpeg | Dummy: acc=0.490, F1=0.000, recall=0.000
QEMU -> FFmpeg | NB: acc=0.496, F1=0.325, recall=0.238
QEMU -> FFmpeg | SVM: acc=0.514, F1=0.475, recall=0.432

FFmpeg -> QEMU | Dummy: acc=0.426, F1=0.598, recall=1.000
FFmpeg -> QEMU | NB: acc=0.483, F1=0.480, recall=0.559
FFmpeg -> QEMU | SVM: acc=0.496, F1=0.457, recall=0.497

FFmpeg -> FFmpeg | Dummy: acc=0.510, F1=0.675, recall=1.000
FFmpeg -> FFmpeg | NB: acc=0.585, F1=0.612, recall=0.642
FFmpeg -> FFmpeg | SVM: acc=0.564, F1=0.584, recall=0.600

QEMU -> QEMU | Dummy: acc=0.574, F1=0.000, recall=0.000
QEMU -> QEMU | NB: acc=0.658, F1=0.585, recall=0.567
QEMU -> QEMU | SVM: acc=0.657, F1=0.611, recall=0.632

